# Chapter 18 — Complete SQL Project Solutions## End-to-End RL Fine-Tuning Walkthrough[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**GPU Required: T4 minimum, A100 recommended.**This notebook provides the complete annotated solution for all 5 stages of the SQL fine-tuning project.

### Stage 1 Solution: SFT Data Preparation and Training

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load Spider dataset
try:
    spider = load_dataset('spider', trust_remote_code=True)
    print(f"Spider loaded: {len(spider['train'])} train, {len(spider['validation'])} val")
except:
    print("Using mock data for demonstration")
    spider = {'train': [{'db_id':'company','question':'How many employees?','query':'SELECT COUNT(*) FROM employees'}]*100,
              'validation': [{'db_id':'company','question':'Who earns most?','query':'SELECT name FROM employees ORDER BY salary DESC LIMIT 1'}]*20}

def format_sft_example(example):
    """Format (question, SQL) as instruction-following example."""
    prompt = (
        f"You are an expert SQL developer. Generate correct, efficient SQL queries.\n\n"
        f"Database: {example['db_id']}\n"
        f"Question: {example['question']}\n\n"
        f"SQL Query:"
    )
    response = " " + example['query']
    return {
        'prompt':   prompt,
        'response': response,
        'text':     prompt + response + tokenizer.eos_token
    }

# Apply formatting
train_formatted = [format_sft_example(ex) for ex in list(spider['train'])[:500]]
val_formatted   = [format_sft_example(ex) for ex in list(spider['validation'])[:50]]

# Check token lengths
lengths = [len(tokenizer(ex['text'])['input_ids']) for ex in train_formatted[:50]]
print(f"\nToken length stats — mean: {sum(lengths)/len(lengths):.0f}, max: {max(lengths)}")
print(f"Sample prompt:\n{train_formatted[0]['prompt']}")
print(f"Sample response:{train_formatted[0]['response']}")

# FULL TRAINING (uncomment on GPU):
# from peft import get_peft_model, LoraConfig, TaskType
# from transformers import AutoModelForCausalLM
# from trl import SFTTrainer, SFTConfig
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto')
# lora = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj','v_proj'], task_type=TaskType.CAUSAL_LM)
# model = get_peft_model(model, lora)
# trainer = SFTTrainer(model=model,
#     args=SFTConfig(output_dir='./sql_sft', num_train_epochs=3, per_device_train_batch_size=4,
#                    gradient_accumulation_steps=4, learning_rate=2e-4, max_seq_length=512),
#     train_dataset=train_formatted, eval_dataset=val_formatted)
# trainer.train()
print("\n[SFT stage ready. Uncomment training code and run on GPU]")


### Stage 2 Solution: Reward Model for SQL Quality

In [ ]:
import re, subprocess, tempfile, os

def sql_execution_reward(predicted_sql: str, gold_sql: str,
                         db_path: str = ':memory:') -> float:
    """
    Multi-component SQL reward:
    1. Syntax validity (does it parse?)
    2. Execution success (does it run without error?)
    3. Result correctness (does it match gold output?)
    """
    import sqlite3

    score = 0.0

    # Component 1: Syntax check (0.1)
    try:
        sqlite3.connect(':memory:').execute(f'EXPLAIN QUERY PLAN {predicted_sql}')
        score += 0.1
    except:
        return 0.0  # Invalid SQL — stop here

    # Component 2: Execution (0.3)
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(predicted_sql)
        predicted_result = set(map(str, cursor.fetchall()))
        score += 0.3
    except Exception as e:
        return score  # Runs but errors

    # Component 3: Correctness — compare with gold (0.6)
    try:
        cursor.execute(gold_sql)
        gold_result = set(map(str, cursor.fetchall()))
        if predicted_result == gold_result:
            score += 0.6  # Exact match
        elif predicted_result & gold_result:
            # Partial credit: some rows match
            overlap = len(predicted_result & gold_result)
            total   = len(predicted_result | gold_result)
            score += 0.6 * (overlap / total)
        conn.close()
    except:
        pass

    return score

# Test reward function
test_cases = [
    ("SELECT COUNT(*) FROM employees",
     "SELECT COUNT(*) FROM employees", "Perfect match"),
    ("SELECT count(*) FROM employees",
     "SELECT COUNT(*) FROM employees", "Case difference (should match)"),
    ("SELECT * FROM employees",
     "SELECT COUNT(*) FROM employees", "Wrong query structure"),
    ("SELECT INVALID SYNTAX!!!",
     "SELECT COUNT(*) FROM employees", "Invalid SQL"),
]

print(f"{'Description':<30} {'Reward':>8}")
print("-" * 42)
for pred, gold, desc in test_cases:
    r = sql_execution_reward(pred, gold)
    print(f"{desc:<30} {r:>8.3f}")

print("\nReward components:")
print("  0.1 — syntax valid (SQL parses without error)")
print("  0.3 — executes without runtime error")
print("  0.6 — result matches gold (with partial credit for overlap)")
print("\nTotal possible: 1.0")
print("Unverifiable quality aspects not rewarded: readability, efficiency")


### Stage 3 Solution: Evaluation Framework

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt

def evaluate_sql_model(model_fn, test_examples, reward_fn, model_name='Model'):
    """Comprehensive evaluation of a SQL generation model."""
    rewards, exact_matches, syntax_valid = [], [], []

    for ex in test_examples:
        try:
            predicted = model_fn(ex['prompt'])
            r = reward_fn(predicted, ex['gold_sql'])
            rewards.append(r)
            exact_matches.append(1.0 if r >= 0.9 else 0.0)
            syntax_valid.append(1.0 if r > 0 else 0.0)
        except Exception as e:
            rewards.append(0.0); exact_matches.append(0.0); syntax_valid.append(0.0)

    return {
        'model': model_name,
        'execution_accuracy': np.mean(rewards),
        'exact_match': np.mean(exact_matches),
        'syntax_valid': np.mean(syntax_valid),
        'n_examples': len(test_examples),
        'reward_distribution': rewards,
    }

# Simulated results (replace with real model outputs on GPU)
np.random.seed(42)
baseline_rewards = np.random.beta(3, 2, 100) * 0.75       # SFT baseline: ~0.65 mean
sft_rewards      = np.random.beta(4, 2, 100) * 0.85       # after SFT: ~0.73 mean
ppo_rewards      = np.random.beta(5, 1.5, 100) * 0.92     # after PPO: ~0.80 mean
dpo_rewards      = np.random.beta(4.5, 1.8, 100) * 0.90  # after DPO: ~0.77 mean

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results = [
    ('Base Model', baseline_rewards, 'gray'),
    ('After SFT',  sft_rewards,      'steelblue'),
    ('SFT + PPO',  ppo_rewards,      'seagreen'),
    ('SFT + DPO',  dpo_rewards,      'purple'),
]

for name, rewards, color in results:
    axes[0].hist(rewards, bins=20, alpha=0.5, label=name, color=color, density=True)

axes[0].set_xlabel('Execution Accuracy'); axes[0].set_ylabel('Density')
axes[0].set_title('Reward Distribution by Training Stage')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Bar chart of mean metrics
means = [np.mean(r) for _, r, _ in results]
names = [n for n,_,_ in results]
colors = [c for _,_,c in results]
axes[1].bar(names, means, color=colors, alpha=0.8)
axes[1].set_ylabel('Mean Execution Accuracy')
axes[1].set_title('Training Stage Comparison')
axes[1].set_ylim(0, 1)
for i, (m, color) in enumerate(zip(means, colors)):
    axes[1].text(i, m+0.01, f'{m:.3f}', ha='center', fontsize=11, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

print("\nExpected results on real Spider dataset:")
print(f"{'Stage':<15} {'Exec Acc':>10} {'Exact Match':>12}")
print("-" * 40)
print(f"{'Base Model':<15} {'~0.52':>10} {'~0.48':>12}")
print(f"{'After SFT':<15} {'~0.73':>10} {'~0.68':>12}")
print(f"{'SFT + PPO':<15} {'~0.81':>10} {'~0.76':>12}")
print(f"{'SFT + DPO':<15} {'~0.77':>10} {'~0.72':>12}")
print("\nKey finding: PPO improves over DPO on verifiable tasks (+4pp)")
print("DPO preferred for simpler tasks (faster, more stable training)")
